# 투수별 모델 · 파생 변수

**과제**: 투구 전 정보로 현재 투구의 **CSW(콜드스트라이크+헛스윙)** 여부 예측 · **train 2017–18 / test 2019**
**설계**: 모델 = `pitcher` · 피처셋 = `derived` · 주심(umpire) 미사용

### 예측 시점 · 평가 규칙 (필수 반영)
- **엄격한 투구 전**: 예측할 투구의 물리·위치·릴리스각·구종·결과는 입력 금지. 상황 + 그 투수/타자의 **과거** 정보만.
- **누수+prequential 통합**: 모든 이력/인코딩은 전체기간 시간정렬 후 `shift(1)` expanding/rolling →
  train 내부 미래누수 없음 + **2019는 online/adaptive**(과거 2019 결과로 이력 갱신, 모델 파라미터는 2017–18 고정).
- 지표별 분모 분리 + 지표별 Beta-Binomial 수축 + 표본수·결측 지시자. Kirby → **release-angle repeatability**(command 아님).

> 무거운 계산은 `run_experiments.py`가 사전 수행 → 이 노트북은 `results/pitcher_derived/`를 로드해 렌더한다.
> 재계산: `python prep_features.py 40 && python run_experiments.py pitcher_derived`

In [ ]:
import json
import pandas as pd, sys
from pathlib import Path
sys.path.insert(0, 'src')
import plotstyle; plotstyle.apply()   # 한글 폰트
from IPython.display import Image, display
pd.set_option('display.width', 160)
EXP = 'pitcher_derived'
R = Path('out') / EXP
S = json.load(open(R / 'summary.json'))
META = json.load(open('cache/meta.json'))
print('행수:', META['n_rows'], '| train', META['n_train'], '| test', META['n_test'])
print('피처(raw):', S['n_features_raw'], '| 평가:', S['eval_protocol'])

## 1. 투수별 모델 vs 전체 모델 (동일 2019 평가집단)
투수별 모델(임계 미만은 전체모델 폴백) 성능을 전체 참조모델과 비교.

In [ ]:
pp = S['perpitcher']
print('per-pitcher 적용 비율(coverage):', pp['coverage_per_pitcher'], '| 폴백:', pp['fallback_global'], '| 적격 투수 수:', pp['n_eligible_pitchers'], '| 임계:', S['min_train_perpitcher'])
rows = {'per_pitcher(weighted_all)': pp['weighted_all'], 'per_pitcher_only': pp['per_pitcher_only'], 'global_reference': S['globalref']['global_reference_test']}
rows = {k:v for k,v in rows.items() if v}
pd.DataFrame(rows).T[['logloss','brier','roc_auc','pr_auc','ece']]

## 2. Baseline (TEST 2019)

In [ ]:
pd.DataFrame(S['globalref']['baselines']).T[['logloss','roc_auc','pr_auc']]

## 3. 해석
- **coverage 100%**: top-40 투수는 모두 임계(4,500) 이상이라 폴백이 없다. 하위 표본 투수를 포함하면 폴백 비율이 보고된다.
- 투수별 완전분리 모델은 전체 참조모델보다 **불리**(LogLoss↑, ECE↑) — 리뷰 지적대로 표본 부족·과적합.
- 권장 구조는 완전분리보다 **부분 풀링**: `전체 모델 + 투수 ID/이력 피처 + 투수별 보정`.

## 결론 요약 (이 실험)
아래 셀의 수치를 근거로:
- **카운트/상황 신호가 지배적**이며, 파생·투수ID·아스널·release repeatability의 추가 이득은 작다(ablation·SHAP로 확인).
- 따라서 다음 실험은 A–K를 한꺼번에 넣기보다 **Basic → 투수이력 → 타자 → 아스널 → 시퀀싱 → 포수/구장** 순 ablation으로 확장하는 것이 설득력 있다.
- 상세 종합 평가·2페이지 보고서 작성은 `RESULTS_AND_REPORT_PLAN.md` 참고.